# Flipkart Gridlock - Remote Inference on Google Colab
This notebook lets you run the YOLOv8s and PaddleOCR detection pipeline on Google Colab's free GPUs. This is useful for testing the models on a large dataset or if you don't have a local GPU.

**Instructions:**
1. Go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run the setup cell below to install dependencies.

In [ ]:
!pip install ultralytics opencv-python-headless paddleocr paddlepaddle-gpu

### Step 2: Download Models
The ultralytics package will automatically download the YOLOv8s model on first run. PaddleOCR will also download its models.

In [ ]:
from ultralytics import YOLO
from paddleocr import PaddleOCR
import cv2
from matplotlib import pyplot as plt
import urllib.request
import numpy as np

# Initialize Models
print("Loading YOLOv8s...")
model = YOLO('yolov8s.pt')

print("Loading PaddleOCR...")
ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=True)
print("Models loaded successfully!")

### Step 3: Run Inference on a Sample Image
Upload an image to Colab or download a sample traffic image.

In [ ]:
# Download a sample traffic image (replace with your own)
sample_url = 'https://images.unsplash.com/photo-1566347065604-06c8b0e89139?q=80&w=1200&auto=format&fit=crop'
urllib.request.urlretrieve(sample_url, 'sample.jpg')

# Read image
image = cv2.imread('sample.jpg')
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# 1. Run YOLO Detection
results = model('sample.jpg')[0]

detections = []
for box in results.boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    class_name = model.names[cls_id]
    
    # Get bounding box coordinates
    x1, y1, x2, y2 = map(int, box.xyxy[0])
    detections.append({'class': class_name, 'conf': conf, 'box': [x1, y1, x2, y2]})
    
    # Draw bounding box
    cv2.rectangle(image_rgb, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(image_rgb, f"{class_name} {conf:.2f}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

# 2. Run OCR (if any cars/motorcycles are detected, we could crop them. Here we run on whole image)
ocr_results = ocr.ocr('sample.jpg', cls=True)
if ocr_results and ocr_results[0]:
    for line in ocr_results[0]:
        bbox, (text, conf) = line
        print(f"Detected Text: {text} (Confidence: {conf:.2f})")

# Show Result
plt.figure(figsize=(12, 8))
plt.imshow(image_rgb)
plt.axis('off')
plt.show()

print(f"Total objects detected: {len(detections)}")
for d in detections:
    print(f"- {d['class']} ({d['conf']:.2f})")